# Unified PDF-to-DOCX Benchmark

**Purpose:** Fair comparison of all 8 PDF-to-DOCX libraries on the **same synthetic PDFs** with the **same ground truth**.

The previous per-library benchmarks had an apples-to-oranges problem: open-source libraries were tested on synthetic PDFs (100% word recall by construction), while Adobe was tested on real PDFs with pypdfium2-extracted ground truth (94-98%). This notebook fixes that.

**Libraries tested:** PyMuPDF, pdfplumber, Camelot, LibreOffice, pdf2docx, Docling, Tesseract, Adobe PDF Services

**Test PDFs:**
| PDF | Features |
|-----|----------|
| plain_text | 10 pages, headings + body paragraphs |
| tables_and_text | 10 pages, text + bordered 5x4 tables |
| logo_and_images | 10 pages, text + embedded PNG images |
| two_column_msa | ~15 pages, two-column legal MSA layout |
| mixed_everything | 10 pages, text + images + tables combined |

## Setup

In [ ]:
import fitz, io, os, re, struct, time, zlib, zipfile, subprocess
from collections import defaultdict
from difflib import SequenceMatcher
from pathlib import Path
from xml.etree import ElementTree as ET

import pypdf
import pdfplumber
os.environ.setdefault("DYLD_LIBRARY_PATH", "/opt/homebrew/lib")
import camelot
import pytesseract
from pdf2image import convert_from_path
from pdf2docx import Converter as Pdf2DocxConverter
from docling.document_converter import DocumentConverter
from htmldocx import HtmlToDocx
from docx import Document
from docx.shared import Inches, Pt
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# Adobe (optional)
try:
    from adobe.pdfservices.operation.auth.service_principal_credentials import ServicePrincipalCredentials
    from adobe.pdfservices.operation.pdf_services import PDFServices
    from adobe.pdfservices.operation.pdf_services_media_type import PDFServicesMediaType
    from adobe.pdfservices.operation.pdfjobs.jobs.export_pdf_job import ExportPDFJob
    from adobe.pdfservices.operation.pdfjobs.params.export_pdf.export_pdf_params import ExportPDFParams
    from adobe.pdfservices.operation.pdfjobs.params.export_pdf.export_pdf_target_format import ExportPDFTargetFormat
    from adobe.pdfservices.operation.pdfjobs.result.export_pdf_result import ExportPDFResult
    ADOBE_AVAILABLE = True
except ImportError:
    ADOBE_AVAILABLE = False

CLIENT_ID = os.environ.get("ADOBE_CLIENT_ID", "")
CLIENT_SECRET = os.environ.get("ADOBE_CLIENT_SECRET", "")
SKIP_ADOBE = not (ADOBE_AVAILABLE and CLIENT_ID and CLIENT_SECRET)

OUTPUT_DIR = "/tmp/unified_benchmark"
PDF_DIR = os.path.join(OUTPUT_DIR, "pdfs")
DOCX_DIR = os.path.join(OUTPUT_DIR, "docx")
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(DOCX_DIR, exist_ok=True)

ground_truths = {}   # pdf_name -> ground truth text string
pdf_paths = {}       # pdf_name -> absolute path to PDF
pdf_metadata = {}    # pdf_name -> {pages, tables_expected, images_expected}
results = []

LIBRARIES = ["PyMuPDF", "pdfplumber", "Camelot", "LibreOffice", "pdf2docx", "Docling", "Tesseract", "Adobe"]

if SKIP_ADOBE:
    print("Adobe SDK not available or credentials not set -- Adobe will be skipped")
    print("  Set ADOBE_CLIENT_ID and ADOBE_CLIENT_SECRET env vars to enable")
else:
    print("Adobe SDK available")

print(f"Output: {OUTPUT_DIR}/")

## Helper Functions

In [ ]:
def create_png(width, height, r, g, b):
    """Create a minimal RGB PNG image in memory."""
    raw_data = b""
    for _y in range(height):
        raw_data += b"\x00"
        for _x in range(width):
            raw_data += bytes([r, g, b])
    def chunk(chunk_type, data):
        c = chunk_type + data
        crc = zlib.crc32(c) & 0xFFFFFFFF
        return struct.pack(">I", len(data)) + c + struct.pack(">I", crc)
    ihdr = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    png = b"\x89PNG\r\n\x1a\n"
    png += chunk(b"IHDR", ihdr)
    png += chunk(b"IDAT", zlib.compress(raw_data))
    png += chunk(b"IEND", b"")
    return png


def extract_text_from_docx(docx_path):
    """Extract ALL text from DOCX including text inside VML/drawing objects.

    Uses XML-level <w:t> parsing rather than python-docx paragraph API,
    because LibreOffice wraps every word in VML text boxes that show as
    0 paragraphs via the API but contain full text in XML.
    """
    with zipfile.ZipFile(docx_path) as z:
        with z.open("word/document.xml") as f:
            tree = ET.parse(f)
    texts = []
    for t_elem in tree.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t"):
        if t_elem.text:
            texts.append(t_elem.text)
    return " ".join(texts)


def compute_similarity(ground_truth, converted_text):
    """Compute text similarity metrics. Same formula for ALL libraries."""
    gt_clean = re.sub(r"\s+", " ", ground_truth).strip()
    cv_clean = re.sub(r"\s+", " ", converted_text).strip()

    # Sequence match (character-level, capped at 50K chars)
    seq_match = SequenceMatcher(None, gt_clean[:50000], cv_clean[:50000]).ratio()

    # Word recall: % of ground truth words found in output
    gt_words = set(gt_clean.lower().split())
    cv_words = set(cv_clean.lower().split())
    word_recall = len(gt_words & cv_words) / max(len(gt_words), 1)

    # Char ratio
    char_ratio = len(cv_clean) / max(len(gt_clean), 1)

    return {
        "seq_match": round(seq_match, 4),
        "word_recall": round(word_recall, 4),
        "char_ratio": round(char_ratio, 2),
    }


def analyze_docx(docx_path):
    """Analyze DOCX for structural metrics."""
    with zipfile.ZipFile(docx_path) as z:
        xml = z.read("word/document.xml").decode()
        vml_boxes = xml.count("v:shape") + xml.count("v:textbox") + xml.count("v:textBox") + xml.count("wps:wsp")
        n_images = len([n for n in z.namelist() if n.startswith("word/media/")])

    doc = Document(docx_path)
    n_paras = len([p for p in doc.paragraphs if p.text.strip()])
    n_tables = len(doc.tables)
    bold_runs = sum(1 for p in doc.paragraphs for r in p.runs if r.bold)

    return {
        "vml_boxes": vml_boxes,
        "paragraphs": n_paras,
        "tables": n_tables,
        "images": n_images,
        "bold_runs": bold_runs,
    }


print("Helper functions ready")

## Generate Test PDFs

In [ ]:
"""PDF 1: Plain Text -- 10 pages of headings + body paragraphs"""
name = "plain_text"
pdf_path = os.path.join(PDF_DIR, f"{name}.pdf")
PAGES = 10

gt_lines = []
doc = fitz.open()
for i in range(PAGES):
    page = doc.new_page(width=612, height=792)
    heading = f"Page {i+1} - Document Heading"
    page.insert_text((72, 72), heading, fontsize=18, fontname="helv")
    gt_lines.append(heading)
    y = 120
    for j in range(20):
        line = (
            f"This is line {j+1} of page {i+1}. Lorem ipsum dolor sit amet, "
            "consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut "
            "labore et dolore magna aliqua. Ut enim ad minim veniam."
        )[:90]
        page.insert_text((72, y), line, fontsize=10, fontname="helv")
        gt_lines.append(line)
        y += 20
doc.save(pdf_path)
doc.close()

ground_truths[name] = "\n".join(gt_lines)
pdf_paths[name] = pdf_path
pdf_metadata[name] = {"pages": PAGES, "tables_expected": 0, "images_expected": 0}
print(f"PDF 1: {name}.pdf -- {PAGES} pages, {os.path.getsize(pdf_path)/1024:.0f} KB, {len(gt_lines)} GT lines")

In [ ]:
"""PDF 2: Tables and Text -- 10 pages with 5x4 bordered tables"""
name = "tables_and_text"
pdf_path = os.path.join(PDF_DIR, f"{name}.pdf")
PAGES = 10

gt_lines = []
doc = fitz.open()
for i in range(PAGES):
    page = doc.new_page(width=612, height=792)
    heading = f"Section {i+1}: Analysis Report"
    page.insert_text((72, 60), heading, fontsize=16, fontname="helv")
    gt_lines.append(heading)
    y = 100
    for j in range(6):
        text = (
            f"Paragraph {j+1}: The quick brown fox jumps over the lazy dog. "
            "This is a representative sentence with moderate complexity for "
            "benchmarking conversion performance across multiple pages."
        )[:95]
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 18
    table_y = y + 10
    for row in range(5):
        for col in range(4):
            x0 = 72 + col * 115
            y0 = table_y + row * 25
            rect = fitz.Rect(x0, y0, x0 + 115, y0 + 25)
            page.draw_rect(rect, color=(0, 0, 0), width=0.5)
            cell_text = f"Cell {row},{col}: data"
            page.insert_text((x0 + 5, y0 + 17), cell_text, fontsize=8, fontname="helv")
            gt_lines.append(cell_text)
    y = table_y + 150
    for j in range(6):
        text = (
            f"Additional content line {j+1}: Further analysis shows that the "
            "data points correlate with the observed trends in the baseline "
            "measurements taken during the study period."
        )[:95]
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 18
doc.save(pdf_path)
doc.close()

ground_truths[name] = "\n".join(gt_lines)
pdf_paths[name] = pdf_path
pdf_metadata[name] = {"pages": PAGES, "tables_expected": PAGES, "images_expected": 0}
print(f"PDF 2: {name}.pdf -- {PAGES} pages, {os.path.getsize(pdf_path)/1024:.0f} KB, {len(gt_lines)} GT lines")

In [ ]:
"""PDF 3: Logo and Images -- 10 pages with embedded PNGs"""
name = "logo_and_images"
pdf_path = os.path.join(PDF_DIR, f"{name}.pdf")
PAGES = 10

logo_bytes = create_png(120, 40, 30, 100, 180)
photo_bytes = create_png(300, 200, 100, 150, 200)

gt_lines = []
doc = fitz.open()
for i in range(PAGES):
    page = doc.new_page(width=612, height=792)
    heading = f"Document Section {i+1}"
    page.insert_text((72, 50), heading, fontsize=16, fontname="helv")
    gt_lines.append(heading)
    y = 80
    for j in range(4):
        text = f"Paragraph {j+1}: Analysis of quarterly performance metrics and KPIs."
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 16
    page.insert_image(fitz.Rect(72, y + 10, 192, y + 50), stream=logo_bytes)
    y += 70
    for j in range(3):
        text = f"Post-logo paragraph {j+1}: Strategic objectives remain on track."
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 16
    page.insert_image(fitz.Rect(72, y + 10, 372, y + 210), stream=photo_bytes)
    y += 230
    for j in range(3):
        text = f"Conclusion {j+1}: Based on the above data we recommend proceeding."
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 16
doc.save(pdf_path)
doc.close()

ground_truths[name] = "\n".join(gt_lines)
pdf_paths[name] = pdf_path
pdf_metadata[name] = {"pages": PAGES, "tables_expected": 0, "images_expected": PAGES * 2}
print(f"PDF 3: {name}.pdf -- {PAGES} pages, {os.path.getsize(pdf_path)/1024:.0f} KB, {len(gt_lines)} GT lines")

In [ ]:
"""PDF 4: Two-Column MSA -- Legal contract in two-column layout (ReportLab)"""
from reportlab.platypus import BaseDocTemplate, Frame, PageTemplate, Paragraph as RLParagraph, Spacer as RLSpacer
from reportlab.lib.pagesizes import letter as rl_letter
from reportlab.lib.units import inch as rl_inch
from reportlab.lib.styles import getSampleStyleSheet as rlGetStyles, ParagraphStyle
from reportlab.lib.enums import TA_JUSTIFY

name = "two_column_msa"
pdf_path = os.path.join(PDF_DIR, f"{name}.pdf")

msa_sections = [
    ("1. DEFINITIONS", [
        '"Agreement" means this Master Service Agreement, including all exhibits and schedules attached hereto.',
        '"Confidential Information" means any information disclosed by one party to the other party, either directly or indirectly, in writing, orally, or by inspection of tangible objects.',
        '"Effective Date" means the date first written above or the date of the last signature below, whichever is later.',
        '"Intellectual Property" means all patents, copyrights, trademarks, trade secrets, and other proprietary rights.',
        '"Services" means the professional services to be provided by Service Provider as described in each Statement of Work.',
        '"Term" means the period commencing on the Effective Date and continuing until terminated in accordance with Section 8.',
    ]),
    ("2. SCOPE OF SERVICES", [
        "Service Provider shall perform the Services described in each Statement of Work executed by both parties.",
        "Each Statement of Work shall describe the scope, timeline, deliverables, fees, and payment terms.",
        "Service Provider shall assign qualified personnel with appropriate skills and experience to perform the Services.",
        "Client may request changes to the Services by submitting a written change order to Service Provider.",
        "Service Provider shall use commercially reasonable efforts to accommodate such change requests.",
    ]),
    ("3. COMPENSATION AND PAYMENT", [
        "Client shall pay Service Provider the fees set forth in each Statement of Work.",
        "Unless otherwise specified, all invoices shall be due and payable within thirty (30) days of receipt.",
        "Late payments shall accrue interest at the rate of one and one-half percent (1.5%) per month.",
        "Service Provider shall submit itemized invoices on a monthly basis detailing the Services performed.",
        "Client shall reimburse Service Provider for reasonable travel and out-of-pocket expenses pre-approved in writing.",
        "All fees are exclusive of applicable taxes, which shall be the responsibility of Client.",
    ]),
    ("4. CONFIDENTIALITY", [
        "Each party agrees to maintain the confidentiality of all Confidential Information received from the other party.",
        "Neither party shall disclose Confidential Information to any third party without prior written consent.",
        "The obligations of confidentiality shall not apply to information that: (a) is or becomes publicly available through no fault of the receiving party; (b) was already known to the receiving party prior to disclosure; (c) is independently developed by the receiving party without use of the Confidential Information; or (d) is required to be disclosed by law or court order.",
        "Upon termination, each party shall return or destroy all Confidential Information in its possession.",
        "The obligations under this Section shall survive termination of this Agreement for a period of five (5) years.",
    ]),
    ("5. INTELLECTUAL PROPERTY", [
        "All pre-existing Intellectual Property shall remain the property of the party that owned it prior to the engagement.",
        "Work product created by Service Provider specifically for Client shall be owned by Client upon full payment.",
        "Service Provider retains the right to use general knowledge, skills, and experience gained during the engagement.",
        "Client grants Service Provider a limited license to use Client materials solely for performing the Services.",
    ]),
    ("6. WARRANTIES AND REPRESENTATIONS", [
        "Service Provider warrants that the Services will be performed in a professional and workmanlike manner.",
        "Service Provider represents that it has the authority to enter into this Agreement and perform the Services.",
        "EXCEPT AS EXPRESSLY SET FORTH HEREIN, NEITHER PARTY MAKES ANY WARRANTIES, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY OR FITNESS FOR A PARTICULAR PURPOSE.",
        "Client warrants that it has the right to provide all materials and information necessary for the Services.",
    ]),
    ("7. LIMITATION OF LIABILITY", [
        "IN NO EVENT SHALL EITHER PARTY BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES.",
        "THE TOTAL LIABILITY OF EITHER PARTY SHALL NOT EXCEED THE FEES PAID OR PAYABLE UNDER THE APPLICABLE STATEMENT OF WORK.",
        "The limitations set forth in this Section shall apply regardless of the form of action, whether in contract, tort, or otherwise.",
        "Nothing in this Agreement shall limit liability for gross negligence, willful misconduct, or fraud.",
    ]),
    ("8. TERM AND TERMINATION", [
        "This Agreement shall commence on the Effective Date and continue for an initial term of one (1) year.",
        "Either party may terminate this Agreement for convenience upon sixty (60) days prior written notice.",
        "Either party may terminate immediately upon material breach if such breach is not cured within thirty (30) days of notice.",
        "Upon termination, Service Provider shall deliver all completed work product to Client.",
        "Sections 4, 5, 7, and 9 shall survive termination of this Agreement.",
    ]),
    ("9. INDEMNIFICATION", [
        "Each party shall indemnify and hold harmless the other party from any claims, damages, or expenses arising from the indemnifying party's breach of this Agreement.",
        "Service Provider shall indemnify Client against claims that the Services infringe any third-party intellectual property rights.",
        "Client shall indemnify Service Provider against claims arising from Client's use of the deliverables in violation of applicable law.",
        "The indemnified party shall provide prompt written notice of any claim and reasonable cooperation in the defense thereof.",
    ]),
    ("10. GENERAL PROVISIONS", [
        "This Agreement constitutes the entire agreement between the parties and supersedes all prior agreements and understandings.",
        "This Agreement may be amended only by a written instrument signed by both parties.",
        "Neither party may assign this Agreement without the prior written consent of the other party.",
        "This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.",
        "Any dispute arising under this Agreement shall be resolved through binding arbitration in Wilmington, Delaware.",
        "If any provision of this Agreement is held invalid or unenforceable, the remaining provisions shall remain in full force.",
        "The waiver of any breach shall not constitute a waiver of any subsequent breach.",
        "All notices shall be in writing and delivered to the addresses set forth on the signature page.",
    ]),
]

gt_lines = []
repeats = 3
for _ in range(repeats):
    for section_title, paragraphs in msa_sections:
        gt_lines.append(section_title)
        for para in paragraphs:
            gt_lines.append(para)

page_w, page_h = rl_letter
margin = 0.75 * rl_inch
col_gap = 0.3 * rl_inch
col_w = (page_w - 2 * margin - col_gap) / 2
frame_left = Frame(margin, margin, col_w, page_h - 2 * margin, id="left")
frame_right = Frame(margin + col_w + col_gap, margin, col_w, page_h - 2 * margin, id="right")
two_col_template = PageTemplate(id="TwoCol", frames=[frame_left, frame_right])

rl_doc = BaseDocTemplate(pdf_path, pagesize=rl_letter)
rl_doc.addPageTemplates([two_col_template])
rl_styles = rlGetStyles()
heading_style = ParagraphStyle("MSAHeading", parent=rl_styles["Heading3"],
    alignment=TA_JUSTIFY, spaceAfter=6, spaceBefore=12, fontSize=10, leading=12, fontName="Helvetica-Bold")
body_style = ParagraphStyle("MSABody", parent=rl_styles["Normal"],
    alignment=TA_JUSTIFY, spaceAfter=6, spaceBefore=2, fontSize=8.5, leading=11, fontName="Times-Roman")

story = []
for _ in range(repeats):
    for section_title, paragraphs in msa_sections:
        story.append(RLParagraph(f"<b>{section_title}</b>", heading_style))
        for para in paragraphs:
            story.append(RLParagraph(para, body_style))
        story.append(RLSpacer(1, 6))
rl_doc.build(story)

with open(pdf_path, "rb") as f:
    content = f.read()
    page_count = content.count(b"/Type /Page") - content.count(b"/Type /Pages")

ground_truths[name] = "\n".join(gt_lines)
pdf_paths[name] = pdf_path
pdf_metadata[name] = {"pages": page_count, "tables_expected": 0, "images_expected": 0}
print(f"PDF 4: {name}.pdf -- {page_count} pages, {os.path.getsize(pdf_path)/1024:.0f} KB, {len(gt_lines)} GT lines")

In [ ]:
"""PDF 5: Mixed Everything -- 10 pages with text + images + tables"""
name = "mixed_everything"
pdf_path = os.path.join(PDF_DIR, f"{name}.pdf")
PAGES = 10

logo_bytes = create_png(120, 40, 30, 100, 180)

gt_lines = []
doc = fitz.open()
for i in range(PAGES):
    page = doc.new_page(width=612, height=792)
    heading = f"Report Section {i+1}"
    page.insert_text((72, 50), heading, fontsize=16, fontname="helv")
    gt_lines.append(heading)
    y = 80
    for j in range(3):
        text = f"Paragraph {j+1}: Strategic planning metrics show continued improvement."
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 16
    page.insert_image(fitz.Rect(72, y + 5, 192, y + 45), stream=logo_bytes)
    y += 60
    table_y = y
    for row in range(4):
        for col in range(3):
            x0 = 72 + col * 150
            y0 = table_y + row * 25
            rect = fitz.Rect(x0, y0, x0 + 150, y0 + 25)
            page.draw_rect(rect, color=(0, 0, 0), width=0.5)
            cell_text = f"Data {row},{col}"
            page.insert_text((x0 + 5, y0 + 17), cell_text, fontsize=8, fontname="helv")
            gt_lines.append(cell_text)
    y = table_y + 120
    for j in range(3):
        text = f"Closing remark {j+1}: All objectives have been met for this period."
        page.insert_text((72, y), text, fontsize=10, fontname="helv")
        gt_lines.append(text)
        y += 16
doc.save(pdf_path)
doc.close()

ground_truths[name] = "\n".join(gt_lines)
pdf_paths[name] = pdf_path
pdf_metadata[name] = {"pages": PAGES, "tables_expected": PAGES, "images_expected": PAGES}
print(f"PDF 5: {name}.pdf -- {PAGES} pages, {os.path.getsize(pdf_path)/1024:.0f} KB, {len(gt_lines)} GT lines")

## Converter Functions

In [ ]:
# Warm up docling models
print("Warming up docling models (first-run model load)...")
from reportlab.platypus import SimpleDocTemplate
from reportlab.lib.pagesizes import letter
_warmup_path = "/tmp/_warmup.pdf"
SimpleDocTemplate(_warmup_path, pagesize=letter).build(
    [RLParagraph("warmup", rlGetStyles()["Normal"])])
docling_converter = DocumentConverter()
_ = docling_converter.convert(_warmup_path)
print("Docling ready.")


def convert_pymupdf(pdf_path, docx_path):
    reader = pypdf.PdfReader(pdf_path)
    doc = Document()
    for page_num, page in enumerate(reader.pages):
        if page_num > 0:
            doc.add_page_break()
        text = page.extract_text() or ""
        for line in text.split("\n"):
            line = line.strip()
            if line:
                doc.add_paragraph(line)
        try:
            for image in page.images:
                try:
                    doc.add_picture(io.BytesIO(image.data), width=Inches(3))
                except Exception:
                    pass
        except Exception:
            pass
    doc.save(docx_path)


def convert_pdfplumber(pdf_path, docx_path):
    doc = Document()
    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            if page_idx > 0:
                doc.add_page_break()
            tables = page.find_tables()
            table_bboxes = [t.bbox for t in tables]
            words = page.extract_words(keep_blank_chars=True, extra_attrs=["size"])
            non_table_words = []
            for w in words:
                inside = False
                for bbox in table_bboxes:
                    if (bbox[0] - 2 <= w["x0"] <= bbox[2] + 2 and
                        bbox[1] - 2 <= w["top"] <= bbox[3] + 2):
                        inside = True
                        break
                if not inside:
                    non_table_words.append(w)
            lines = {}
            for w in non_table_words:
                key = round(w["top"], 1)
                if key not in lines:
                    lines[key] = []
                lines[key].append(w)
            content_items = []
            for y in sorted(lines.keys()):
                lw = sorted(lines[y], key=lambda w: w["x0"])
                content_items.append((y, "text", " ".join(w["text"] for w in lw)))
            for table in tables:
                content_items.append((table.bbox[1], "table", table.extract()))
            content_items.sort(key=lambda x: x[0])
            for _, item_type, data in content_items:
                if item_type == "text":
                    doc.add_paragraph(data)
                elif item_type == "table":
                    rows = data
                    if not rows or not rows[0]:
                        continue
                    num_rows = len(rows)
                    num_cols = max(len(r) for r in rows)
                    tbl = doc.add_table(rows=num_rows, cols=num_cols, style="Table Grid")
                    for i, row in enumerate(rows):
                        for j, val in enumerate(row):
                            if j < num_cols:
                                tbl.cell(i, j).text = val or ""
    doc.save(docx_path)


def convert_camelot(pdf_path, docx_path):
    doc_pdf = fitz.open(pdf_path)
    all_tables = {}
    try:
        tables = camelot.read_pdf(pdf_path, pages="all", flavor="stream", suppress_stdout=True)
        for t in tables:
            all_tables.setdefault(t.page, []).append(t)
    except Exception:
        pass
    doc = Document()
    for page_idx in range(len(doc_pdf)):
        if page_idx > 0:
            doc.add_page_break()
        page = doc_pdf[page_idx]
        blocks = page.get_text("blocks")
        text_blocks = [(b[1], b[4].strip()) for b in blocks if b[6] == 0 and b[4].strip()]
        for _, text in sorted(text_blocks, key=lambda x: x[0]):
            for line in text.split("\n"):
                line = line.strip()
                if line:
                    doc.add_paragraph(line)
        for t in all_tables.get(page_idx + 1, []):
            df = t.df
            tbl = doc.add_table(rows=df.shape[0], cols=df.shape[1], style="Table Grid")
            for i, row in df.iterrows():
                for j, val in enumerate(row):
                    tbl.rows[i].cells[j].text = str(val)
        for img_info in page.get_images(full=True):
            try:
                base_image = doc_pdf.extract_image(img_info[0])
                if base_image:
                    doc.add_picture(io.BytesIO(base_image["image"]), width=Inches(3))
            except Exception:
                pass
    doc_pdf.close()
    doc.save(docx_path)


def convert_libreoffice(pdf_path, docx_path):
    out_dir = os.path.dirname(docx_path)
    subprocess.run(
        ["soffice", "--headless", "--infilter=writer_pdf_import",
         "--convert-to", "docx", "--outdir", out_dir, pdf_path],
        capture_output=True, text=True, timeout=120)
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    generated = os.path.join(out_dir, base + ".docx")
    if generated != docx_path and os.path.exists(generated):
        os.rename(generated, docx_path)


def convert_pdf2docx(pdf_path, docx_path):
    cv = Pdf2DocxConverter(pdf_path)
    cv.convert(docx_path)
    cv.close()


def convert_docling(pdf_path, docx_path):
    result = docling_converter.convert(pdf_path)
    html = result.document.export_to_html()
    doc = Document()
    HtmlToDocx().add_html_to_document(html, doc)
    doc.save(docx_path)


def convert_tesseract(pdf_path, docx_path):
    images = convert_from_path(pdf_path, dpi=200)
    doc = Document()
    for i, img in enumerate(images):
        if i > 0:
            doc.add_page_break()
        text = pytesseract.image_to_string(img)
        for line in text.split("\n"):
            line = line.strip()
            if line:
                doc.add_paragraph(line)
    doc.save(docx_path)


def convert_adobe(pdf_path, docx_path):
    creds = ServicePrincipalCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
    svc = PDFServices(credentials=creds)
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()
    asset = svc.upload(input_stream=io.BytesIO(pdf_bytes), mime_type=PDFServicesMediaType.PDF)
    params = ExportPDFParams(target_format=ExportPDFTargetFormat.DOCX)
    job = ExportPDFJob(input_asset=asset, export_pdf_params=params)
    loc = svc.submit(job)
    resp = svc.get_job_result(loc, ExportPDFResult)
    stream = svc.get_content(resp.get_result().get_asset())
    out = stream.get_input_stream()
    docx_bytes = out if isinstance(out, bytes) else out.read()
    with open(docx_path, "wb") as f:
        f.write(docx_bytes)


CONVERTERS = {
    "PyMuPDF": convert_pymupdf,
    "pdfplumber": convert_pdfplumber,
    "Camelot": convert_camelot,
    "LibreOffice": convert_libreoffice,
    "pdf2docx": convert_pdf2docx,
    "Docling": convert_docling,
    "Tesseract": convert_tesseract,
    "Adobe": convert_adobe,
}

print("All converter functions defined")

## Run Benchmark

In [ ]:
print("=" * 80)
print("RUNNING UNIFIED BENCHMARK")
print("=" * 80)

for pdf_name in pdf_paths:
    pdf_path = pdf_paths[pdf_name]
    gt_text = ground_truths[pdf_name]
    meta = pdf_metadata[pdf_name]

    print(f"\n{'─' * 60}")
    print(f"PDF: {pdf_name} ({meta['pages']} pages)")
    print(f"{'─' * 60}")

    for lib_name in LIBRARIES:
        if lib_name == "Adobe" and SKIP_ADOBE:
            print(f"  [{lib_name:12s}] SKIPPED (no credentials)")
            results.append({
                "pdf": pdf_name, "library": lib_name, "time_s": None,
                "word_recall": None, "seq_match": None, "char_ratio": None,
                "vml_boxes": None, "tables": None, "images": None,
                "paragraphs": None, "success": False, "error": "skipped",
            })
            continue

        docx_path = os.path.join(DOCX_DIR, f"{pdf_name}_{lib_name}.docx")

        start = time.time()
        error_msg = ""
        try:
            CONVERTERS[lib_name](pdf_path, docx_path)
            elapsed = time.time() - start
            success = os.path.exists(docx_path) and os.path.getsize(docx_path) > 0
        except Exception as e:
            elapsed = time.time() - start
            success = False
            error_msg = str(e)[:100]

        if success:
            try:
                extracted_text = extract_text_from_docx(docx_path)
                similarity = compute_similarity(gt_text, extracted_text)
                analysis = analyze_docx(docx_path)

                results.append({
                    "pdf": pdf_name, "library": lib_name,
                    "time_s": round(elapsed, 2),
                    "word_recall": similarity["word_recall"],
                    "seq_match": similarity["seq_match"],
                    "char_ratio": similarity["char_ratio"],
                    "vml_boxes": analysis["vml_boxes"],
                    "tables": analysis["tables"],
                    "images": analysis["images"],
                    "paragraphs": analysis["paragraphs"],
                    "success": True,
                })
                print(f"  [{lib_name:12s}] {elapsed:6.1f}s  word_recall={similarity['word_recall']:.1%}  seq={similarity['seq_match']:.1%}  tables={analysis['tables']}  imgs={analysis['images']}  vml={analysis['vml_boxes']}")
            except Exception as e:
                results.append({
                    "pdf": pdf_name, "library": lib_name,
                    "time_s": round(elapsed, 2),
                    "word_recall": None, "seq_match": None, "char_ratio": None,
                    "vml_boxes": None, "tables": None, "images": None,
                    "paragraphs": None, "success": False,
                    "error": f"analysis failed: {str(e)[:80]}",
                })
                print(f"  [{lib_name:12s}] {elapsed:6.1f}s  ANALYSIS ERROR: {str(e)[:60]}")
        else:
            results.append({
                "pdf": pdf_name, "library": lib_name,
                "time_s": round(elapsed, 2),
                "word_recall": None, "seq_match": None, "char_ratio": None,
                "vml_boxes": None, "tables": None, "images": None,
                "paragraphs": None, "success": False, "error": error_msg,
            })
            print(f"  [{lib_name:12s}] {elapsed:6.1f}s  FAILED: {error_msg[:60]}")

print(f"\n{'=' * 80}")
print(f"BENCHMARK COMPLETE: {len(results)} results")
print(f"{'=' * 80}")

## Results

### Word Recall
The **key metric** -- same PDFs, same ground truth, same formula for all libraries.

`word_recall = len(gt_words & output_words) / len(gt_words)`

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

# Word Recall Table
print("=" * 80)
print("WORD RECALL -- % of ground truth words found in converted DOCX")
print("(Same PDFs, same ground truth, same formula for all libraries)")
print("=" * 80)
print()

pivot_wr = df.pivot_table(index="library", columns="pdf", values="word_recall", aggfunc="first")
pivot_wr["AVERAGE"] = pivot_wr.mean(axis=1)
pivot_wr = pivot_wr.sort_values("AVERAGE", ascending=False)
print(pivot_wr.map(lambda x: f"{x:.1%}" if pd.notna(x) else "  --").to_string())

print()
print()

# Speed Table
print("=" * 80)
print("SPEED -- Conversion time in seconds")
print("=" * 80)
print()

pivot_time = df.pivot_table(index="library", columns="pdf", values="time_s", aggfunc="first")
pivot_time["AVERAGE"] = pivot_time.mean(axis=1)
pivot_time = pivot_time.sort_values("AVERAGE")
print(pivot_time.map(lambda x: f"{x:.1f}s" if pd.notna(x) else "  --").to_string())

print()
print()

# Structural Metrics
print("=" * 80)
print("STRUCTURAL METRICS")
print("=" * 80)

for pdf_name in pdf_paths:
    meta = pdf_metadata[pdf_name]
    print(f"\n  {pdf_name} (expected: {meta['tables_expected']} tables, {meta['images_expected']} images)")
    pdf_df = df[df["pdf"] == pdf_name].sort_values("library")
    for _, row in pdf_df.iterrows():
        if row["success"]:
            print(f"    {row['library']:12s}  tables={row['tables']:>3}  images={row['images']:>3}  vml={row['vml_boxes']:>4}")
        else:
            label = "SKIPPED" if row.get("error") == "skipped" else "FAILED"
            print(f"    {row['library']:12s}  {label}")

## Charts

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({"font.size": 10, "figure.dpi": 120})

successful = df[df["success"] == True].copy()
libs_order = successful.groupby("library")["word_recall"].mean().sort_values(ascending=False).index.tolist()
pdfs_order = list(pdf_paths.keys())

# Chart 1: Word Recall Heatmap
fig, ax = plt.subplots(figsize=(12, 6))
pivot = successful.pivot_table(index="library", columns="pdf", values="word_recall", aggfunc="first")
pivot = pivot.reindex(index=libs_order, columns=pdfs_order)

im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0.8, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(pdfs_order)))
ax.set_xticklabels(pdfs_order, rotation=30, ha="right")
ax.set_yticks(range(len(libs_order)))
ax.set_yticklabels(libs_order)
for i in range(len(libs_order)):
    for j in range(len(pdfs_order)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1%}", ha="center", va="center", fontsize=9,
                    color="white" if val < 0.9 else "black")
plt.colorbar(im, label="Word Recall")
ax.set_title("Word Recall by Library x PDF\n(Same ground truth for all libraries)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "word_recall_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

# Chart 2: Speed Comparison
fig, ax = plt.subplots(figsize=(12, 6))
pivot_t = successful.pivot_table(index="library", columns="pdf", values="time_s", aggfunc="first")
avg_time = pivot_t.mean(axis=1).sort_values()
libs_speed = avg_time.index.tolist()
pivot_t = pivot_t.reindex(index=libs_speed, columns=pdfs_order)

x = np.arange(len(libs_speed))
width = 0.15
colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]
for i, pdf_name in enumerate(pdfs_order):
    vals = [pivot_t.loc[lib, pdf_name] if lib in pivot_t.index and pd.notna(pivot_t.loc[lib, pdf_name]) else 0
            for lib in libs_speed]
    ax.barh(x + i * width - width * len(pdfs_order) / 2, vals, width,
            label=pdf_name, color=colors[i % len(colors)], edgecolor="#333", linewidth=0.3, alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels(libs_speed)
ax.set_xlabel("Time (seconds)")
ax.set_title("Conversion Speed by Library x PDF", fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "speed_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nCharts saved to {OUTPUT_DIR}/")